# Fase 5 — Execução do treinamento

**Objetivo da fase:** treinar o adaptador LoRA com a configuração da Fase 4, acompanhar as curvas de loss de treino e de validação e salvar o adaptador no Google Drive para a avaliação da Fase 6 e a demo da Fase 7.

## 5.1 Preparar o ambiente

### 5.1.1 Configurar o Google Colab

Abrir o link do notebook e ir em *Arquivo → Salvar uma cópia no Drive*. A partir daqui, trabalhar sempre na cópia.

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **GPU T4** e **Versão 2026.10**, a mesma da Fase 1. Esta é a fase que mais usa a GPU: o treino completo leva alguns minutos na T4. Usar um **runtime novo**, sem nada da Fase 4 na memória: tudo o que o treino precisa vem do Hub e do Drive.

O secret `HF_TOKEN` criado na Fase 1 já fica disponível para todos os notebooks da conta. No painel lateral, ícone de chave, ligar *Acesso ao notebook* para o `HF_TOKEN` e rodar a célula abaixo para verificar a configuração.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

### 5.1.2 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo. São as mesmas da Fase 4, mais o `tensorboard` para acompanhar as curvas durante o treino:

In [ ]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q transformers==5.22.0 datasets==4.8.5 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 huggingface_hub==1.33.0 tensorboard==2.21.0
# O runtime do Colab traz o torchao 0.10.0, que o peft recusa ao ser importado. O projeto não usa o torchao (não há quantização)
!pip uninstall -q -y torchao

## 5.2 Carregar a configuração e o dataset

### 5.2.1 Carregar a configuração da Fase 4

Montar o Google Drive e carregar o JSON salvo na Fase 4. O LoRA e os hiperparâmetros vêm do arquivo, sem copiar valores à mão, então o treino usa **exatamente** a configuração que passou no teste de fumaça.

Ao rodar a célula, o Colab pede permissão para acessar o Google Drive da conta:

In [ ]:
import json
from google.colab import drive
from peft import LoraConfig
from trl import SFTConfig

drive.mount("/content/drive")

PROJETO_DIR = "/content/drive/MyDrive/fine-tuning-soc"
CONFIG_PATH = f"{PROJETO_DIR}/config_treino.json"

with open(CONFIG_PATH) as f:
    config_treino = json.load(f)

lora_config = LoraConfig(**config_treino["lora"])  # Fase 4, seção 4.3.3
sft_config = SFTConfig(**config_treino["sft"])     # Fase 4, seção 4.4.1
print(json.dumps(config_treino, ensure_ascii=False, indent=2))
# Esperado: os mesmos valores salvos na seção 4.6 da Fase 4

### 5.2.2 Carregar o dataset conversacional

Carregar o dataset no formato prompt/completion publicado na Fase 3. O id vem da configuração:

In [ ]:
from datasets import load_dataset

conversas = load_dataset(config_treino["sft_id"])
print(conversas)
# Esperado: train com 1033, validation com 224 e test com 221 exemplos

O split de **teste** não é usado nesta fase. O treino só vê o `train`, e o `validation` serve para acompanhar o *overfitting* durante o treino. O teste fica intocado para a medição da Fase 6. Se ele fosse usado para decidir alguma coisa do treino, a métrica final deixaria de medir dados realmente novos.

## 5.3 Treino com LoRA + TRL

### 5.3.1 Montar o `SFTTrainer`

Carregar o modelo base em `fp16` e montar o trainer como no teste de fumaça da Fase 4, agora com a configuração completa:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer

MODEL_ID = config_treino["model_id"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=conversas["train"],
    eval_dataset=conversas["validation"],
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()
# Esperado: cerca de 2,2 milhões de parâmetros treináveis, menos de 0,5% do total

### 5.3.2 Acompanhar no TensorBoard (opcional)

A configuração usa `report_to="tensorboard"`, que grava as métricas em `outputs/`. Abrir o painel **antes** de treinar e clicar no botão de atualizar durante o treino para ver as curvas crescendo:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs

### 5.3.3 Treinar

Treinar os 130 passos calculados na Fase 4. O loss de treino é registrado a cada 10 passos e o de validação a cada 20. Ao final, avaliar mais uma vez para ter o loss de validação também no último passo:

In [ ]:
import math

metricas = trainer.train()
avaliacao_final = trainer.evaluate()

print(f"Passos:                    {trainer.state.global_step}")
print(f"Tempo de treino:           {metricas.metrics['train_runtime'] / 60:.1f} min")
print(f"Loss médio de treino:      {metricas.training_loss:.3f}")
print(f"Loss de validação (final): {avaliacao_final['eval_loss']:.3f}")
print(f"Pico de memória da GPU:    {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
assert math.isfinite(avaliacao_final["eval_loss"]), "loss de validação NaN ou infinito"
# Esperado: 130 passos

## 5.4 Leitura das curvas de loss

### 5.4.1 Plotar as curvas

O `trainer.state.log_history` guarda todos os registros do treino. Separar o loss de treino e o de validação e plotar os dois no mesmo gráfico:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

historico = pd.DataFrame(trainer.state.log_history)
treino = historico.dropna(subset=["loss"])[["step", "loss"]]
validacao = historico.dropna(subset=["eval_loss"])[["step", "eval_loss"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(treino["step"], treino["loss"], marker="o", color="tab:blue", label="treino")
ax.plot(validacao["step"], validacao["eval_loss"], marker="s", color="tab:orange", label="validação")
ax.set_xlabel("Passo")
ax.set_ylabel("Loss")
ax.set_title("Curvas de loss do treino")
ax.legend()
plt.show()

### 5.4.2 Diagnosticar

Resumir as duas curvas e localizar o menor loss de validação:

In [ ]:
melhor = validacao.loc[validacao["eval_loss"].idxmin()]

print(f"Loss de treino:    {treino['loss'].iloc[0]:.3f} → {treino['loss'].iloc[-1]:.3f}")
print(f"Loss de validação: {validacao['eval_loss'].iloc[0]:.3f} → {validacao['eval_loss'].iloc[-1]:.3f}")
print(f"Menor validação:   {melhor['eval_loss']:.3f} no passo {int(melhor['step'])} de {trainer.state.global_step}")
validacao.set_index("step").round(3)

Comparar o comportamento das curvas com a tabela abaixo:

| Padrão observado | Diagnóstico | Ação |
|---|---|---|
| Treino ↓ e validação ↓ | Aprendizado saudável | Manter |
| Treino ↓ e validação ↑ ou estagnada | **Overfitting**: o modelo decorou os exemplos (hosts, usuários, frases do template) em vez de aprender a política | Menos épocas, mais `lora_dropout`, menor `r` |
| Ambas altas e estagnadas | Underfitting | Maior `learning_rate` ou `r`. Conferir máscara e template (Fase 3) |
| Oscilando ou `NaN` | Instabilidade | Menor `learning_rate`, conferir `fp16` |

**Leitura:**

- O loss de treino cai rápido nos primeiros passos: o modelo aprende primeiro o **formato** (as chaves do JSON e o `<|im_end|>`), que é igual em todos os exemplos. Depois a queda fica mais lenta, quando o que resta aprender é a política de severidade e o texto da justificativa.
- Se a menor validação está no último passo ou perto dele, a validação ainda acompanha o treino e não há sinal de *overfitting*.
- Se a validação passa a subir enquanto o treino continua caindo, anotar o passo em que isso começou. Esse é o sintoma da linha de *overfitting*, e deve ser citado na conclusão da Fase 6.
- O loss de validação é um sinal de tendência, não a meta. Um loss baixo não garante recall alto de "crítica". Quem decide se o modelo serve são as métricas da Fase 6, no conjunto de teste.

O projeto não tem iteração: o adaptador treinado aqui é a versão final, avaliada na Fase 6 e servida na Fase 7. A coluna **Ação** da tabela fica registrada como o caminho para uma próxima versão, e não é aplicada neste projeto.

## 5.5 Salvar o adaptador no Google Drive

### 5.5.1 Salvar

Salvar o **adaptador LoRA** e o tokenizador na pasta do projeto no Drive, ao lado da configuração da Fase 4:

In [ ]:
import os

ADAPTER_DIR = f"{PROJETO_DIR}/soc-triage-qwen2.5-0.5b-lora"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

for nome in sorted(os.listdir(ADAPTER_DIR)):
    print(f"{nome:30} {os.path.getsize(f'{ADAPTER_DIR}/{nome}') / 1e6:6.1f} MB")
# Esperado: adapter_model.safetensors com poucos MB, adapter_config.json e os arquivos do tokenizador

- Só o adaptador é salvo, com poucos MB. O modelo base continua no Hub e é carregado por baixo do adaptador na inferência.
- Salvar no Drive logo após o treino é a cópia de segurança: se o runtime do Colab desconectar, a Fase 6 e a Fase 7 recarregam o adaptador do Drive sem precisar treinar de novo.

### 5.5.2 Conferir o adaptador salvo

Recarregar o adaptador **do Drive** sobre um modelo base novo, do mesmo jeito que as Fases 6 e 7 vão fazer, e gerar a triagem de um exemplo da validação. Para comparar, gerar também com o adaptador desligado (`disable_adapter`), que é o modelo base recebendo o mesmo prompt de sistema curto:

In [ ]:
from peft import PeftModel

tokenizer_salvo = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
modelo_salvo = PeftModel.from_pretrained(base, ADAPTER_DIR)
modelo_salvo.eval()


def gerar(modelo, prompt: list[dict]) -> str:
    inputs = tokenizer_salvo.apply_chat_template(
        prompt, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        out = modelo.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer_salvo.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


ex = conversas["validation"][0]
print("--- Alerta ---")
print(ex["prompt"][1]["content"])
print("\n--- Esperado ---")
print(ex["completion"][0]["content"])
print("\n--- Com o adaptador ---")
print(gerar(modelo_salvo, ex["prompt"]))
with modelo_salvo.disable_adapter():
    print("\n--- Sem o adaptador (modelo base) ---")
    print(gerar(modelo_salvo, ex["prompt"]))

**Leitura:** com o adaptador, a saída deve ser um JSON com as 3 chaves, no mesmo formato da triagem esperada, e terminar logo depois do JSON. Sem o adaptador, o modelo base recebe só o prompt de sistema curto, sem a tabela da política nem exemplos, e costuma errar o formato, as categorias ou a severidade. A diferença entre as duas saídas é o que o treino ensinou.

Um exemplo só confere que o adaptador foi salvo e recarregado corretamente. Ele não mede desempenho: a medição em todo o teste, com as metas da Fase 1, é feita na Fase 6.

## 5.6 Conclusão

Resultado do treinamento:

| Verificação | Resultado esperado | Onde conferir |
|---|---|---|
| Configuração | A mesma da Fase 4, carregada de `config_treino.json` | Seção 5.2.1 |
| Parâmetros treináveis | Cerca de 2,2 milhões, menos de 0,5% do modelo | Seção 5.3.1 |
| Duração | 130 passos, com loss de treino a cada 10 e validação a cada 20 | Seção 5.3.3 |
| Curvas de loss | Treino e validação caindo juntos, sem `NaN` | Seções 5.4.1 e 5.4.2 |
| Adaptador | Poucos MB, salvo em `fine-tuning-soc/soc-triage-qwen2.5-0.5b-lora` no Drive | Seção 5.5.1 |
| Recarga do Drive | Triagem em JSON com o adaptador, diferente da saída do modelo base | Seção 5.5.2 |

**Conclusão:** o adaptador LoRA foi treinado com a configuração testada na Fase 4, e as curvas de loss mostram se o modelo aprendeu a política ou só decorou os exemplos de treino. O adaptador está salvo no Drive e foi recarregado sobre um modelo base novo, como será feito nas próximas fases. A Fase 6 carrega esse adaptador, gera a triagem para todo o conjunto de teste e compara com o baseline da Fase 1.